# 05 Explainability (Day 10)

Steps in this notebook: SHAP explanation (this part), then calibration and threshold, error analysis, subgroup audit, and the one-time test evaluation.

All analysis uses the **validation split**. The test split is not touched in this part.

Setup: run from the repo root so `src` and `data/` resolve.

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print(os.getcwd())

Imports.

In [ ]:
import joblib  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import shap  # noqa: E402

from src.pipeline import (  # noqa: E402
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    load_splits,
    split_xy,
)

os.makedirs("reports/figures", exist_ok=True)

Load the saved model and the validation split (the test split is ignored here).

In [ ]:
model = joblib.load("models/best_model.joblib")
_train, val, _test = load_splits()
val_x, val_y = split_xy(val)
y_val = val_y.to_numpy()
proba = model.predict_proba(val_x)[:, 1]
print(val_x.shape, "positives:", int(y_val.sum()))

Split the pipeline into its preprocessing and XGBoost parts, so SHAP sees the model's real input columns with readable names.

In [ ]:
prep = model.named_steps["prep"]
clf = model.named_steps["clf"]
x_val_t = np.asarray(prep.transform(val_x), dtype=float)
names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
print(x_val_t.shape, names[:6])

Compute exact TreeSHAP values (log-odds units) for every validation row.

In [ ]:
explainer = shap.TreeExplainer(clf)
sv = explainer.shap_values(x_val_t)
print(sv.shape)

Beeswarm summary plot: each dot is one row, colour is the feature value, position is its push on the prediction. Saved to `reports/figures/shap_summary.png`.

In [ ]:
shap.summary_plot(sv, x_val_t, feature_names=names, max_display=15, show=False)
plt.savefig("reports/figures/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

Mean absolute SHAP bar chart for the top 15 model inputs. Saved to `reports/figures/shap_importance.png`.

In [ ]:
shap.summary_plot(
    sv, x_val_t, feature_names=names, plot_type="bar", max_display=15, show=False
)
plt.savefig("reports/figures/shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()

Importance per original column: adds up each column's one-hot pieces per row first, so the totals are exact (this answers Gate 2 probe 2: which feature is strongest).

In [ ]:
def original_column(name):
    for col in NUMERIC_FEATURES + CATEGORICAL_FEATURES:
        if name == col or name.startswith(col + "_"):
            return col
    return name


groups = {}
for i, n in enumerate(names):
    groups.setdefault(original_column(n), []).append(i)
by_column = (
    pd.Series({c: np.abs(sv[:, idx].sum(axis=1)).mean() for c, idx in groups.items()})
    .sort_values(ascending=False)
    .round(4)
)
by_column.to_csv("reports/shap_by_column.csv", header=["mean_abs_shap"])
by_column

Direction check for `error_code`: for each code, how many rows have it and whether it pushes the anomaly score up (positive) or down (negative).

In [ ]:
rows = []
for i, n in enumerate(names):
    if n.startswith("error_code_"):
        active = x_val_t[:, i] == 1
        if active.sum() > 0:
            rows.append(
                {
                    "category": n,
                    "n_rows": int(active.sum()),
                    "mean_shap_when_active": round(float(sv[active, i].mean()), 4),
                }
            )
pd.DataFrame(rows).sort_values("mean_shap_when_active", ascending=False)

Waterfall plots for 3 single predictions: the most confident true positive, the most confident false positive, and a missed anomaly (false negative). Saved to `reports/figures/`.

In [ ]:
pos = np.where(y_val == 1)[0]
neg = np.where(y_val == 0)[0]
picks = {
    "true_positive": pos[np.argmax(proba[pos])],
    "false_positive": neg[np.argmax(proba[neg])],
    "false_negative": pos[np.argmin(proba[pos])],
}
base_value = float(np.ravel(explainer.expected_value)[0])
for label, i in picks.items():
    exp = shap.Explanation(
        values=sv[i],
        base_values=base_value,
        data=x_val_t[i],
        feature_names=names,
    )
    shap.plots.waterfall(exp, max_display=12, show=False)
    plt.title(f"{label} (row {i}, score {proba[i]:.3f})")
    plt.savefig(f"reports/figures/shap_waterfall_{label}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(label, "row", int(i), "score", round(float(proba[i]), 4))